### 医学影像数据可视化裁剪

- 参数设置

In [ ]:
med_img_path = "data/test_img.nii"
cropped_save_path = "data/med_img_cropped.nii.gz"

- 函数定义

In [ ]:
import SimpleITK as sitk
from scipy.ndimage import zoom
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Load images and get metadata of Size, Origin, Spacing
def get_image_info(path):
    img = sitk.ReadImage(path)
    return img, {
        "shape": img.GetSize(),
        "origin": img.GetOrigin(),
        "spacing": img.GetSpacing(),
    }

# set isotropy of 3D images if needed
def set_isotropy(arr, spacing, new_spacing=1.0):
    zoom_factors = [s / new_spacing  for s in spacing]
    return zoom(arr, zoom_factors, order=1)

# Visualize center slices in three directions of a 3D image (axial, coronal, sagittal)
def visualize_slices(arr, cmap='gray', has_colorbar=False):
    import matplotlib.pyplot as plt
    mid_slices = [arr.shape[0] // 2, arr.shape[1] // 2, arr.shape[2] // 2]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    # Axial
    im0 = axes[0].imshow(arr[mid_slices[0], :, :], cmap=cmap)
    axes[0].set_xlabel('X')
    axes[0].set_ylabel('Y')
    if has_colorbar:
        fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
    # Coronal
    im1 = axes[1].imshow(arr[:, mid_slices[1], :], cmap=cmap)
    axes[1].set_xlabel('X')
    axes[1].set_ylabel('Z')
    if has_colorbar:
        fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    # Sagittal
    im2 = axes[2].imshow(arr[:, :, mid_slices[2]], cmap=cmap)
    axes[2].set_xlabel('Y')
    axes[2].set_ylabel('Z')
    if has_colorbar:
        fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

# Visualize slices at a specified position in three directions (axial, coronal, sagittal) with intersection lines and color coding
def visualize_slices_at(arr, position, cmap='gray', add_line=True, line_width=1.5, highlight_center=False, has_labels=True):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Circle

    z, y, x = position
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Define colors for different planes and markers
    c_axial = 'tab:red'
    c_coronal = 'tab:green'
    c_sagittal = 'tab:cyan'

    # Set radius for highlighting the center point (in data coordinates)
    radius = max(arr.shape) * 0.01

    # Axial: arr[z, :, :] -> (Y, X)
    axes[0].imshow(arr[z, :, :], cmap=cmap)
    if add_line:
        axes[0].axhline(y=y, color=c_coronal, linestyle='--', linewidth=line_width)
        axes[0].axvline(x=x, color=c_sagittal, linestyle='--', linewidth=line_width)
        axes[0].plot(x, y, marker='+', color=c_axial, markersize=10, markeredgewidth=2)
    if highlight_center:
        axes[0].add_patch(Circle((x, y), radius=radius, edgecolor=c_axial, facecolor='none', linewidth=line_width*1.5))
    axes[0].set_title('Axial Slice (Z={})'.format(z))
    if has_labels:
        axes[0].plot([], [], color=c_coronal, linestyle='--', label=f'y={y}')
        axes[0].plot([], [], color=c_sagittal, linestyle='--', label=f'x={x}')
        axes[0].plot([], [], marker='+', color=c_axial, label=f'z={z}')
        axes[0].legend(loc='upper right', fontsize='small')

    # Coronal: arr[:, y, :] -> (Z, X)
    axes[1].imshow(arr[:, y, :], cmap=cmap)
    if add_line:
        axes[1].axhline(y=z, color=c_axial, linestyle='--', linewidth=line_width)
        axes[1].axvline(x=x, color=c_sagittal, linestyle='--', linewidth=line_width)
        axes[1].plot(x, z, marker='+', color=c_coronal, markersize=10, markeredgewidth=2)
    if highlight_center:
        axes[1].add_patch(Circle((x, z), radius=radius, edgecolor=c_coronal, facecolor='none', linewidth=line_width*1.5))
    axes[1].set_title('Coronal Slice (Y={})'.format(y))
    if has_labels:
        axes[1].plot([], [], color=c_axial, linestyle='--', label=f'z={z}')
        axes[1].plot([], [], color=c_sagittal, linestyle='--', label=f'x={x}')
        axes[1].plot([], [], marker='+', color=c_coronal, label=f'y={y}')
        axes[1].legend(loc='upper right', fontsize='small')

    # Sagittal: arr[:, :, x] -> (Z, Y)
    axes[2].imshow(arr[:, :, x], cmap=cmap)
    if add_line:
        axes[2].axhline(y=z, color=c_axial, linestyle='--', linewidth=line_width)
        axes[2].axvline(x=y, color=c_coronal, linestyle='--', linewidth=line_width)
        axes[2].plot(y, z, marker='+', color=c_sagittal, markersize=10, markeredgewidth=2)
    if highlight_center:
        axes[2].add_patch(Circle((y, z), radius=radius, edgecolor=c_sagittal, facecolor='none', linewidth=line_width*1.5))
    axes[2].set_title('Sagittal Slice (X={})'.format(x))
    if has_labels:
        axes[2].plot([], [], color=c_axial, linestyle='--', label=f'z={z}')
        axes[2].plot([], [], color=c_coronal, linestyle='--', label=f'y={y}')
        axes[2].plot([], [], marker='+', color=c_sagittal, label=f'x={x}')
        axes[2].legend(loc='upper right', fontsize='small')

    plt.tight_layout()
    plt.show()


# Interactive visualization of slices at a specified position in three directions (axial, coronal, sagittal) with sliders to adjust the position
def visualize_slices_at_interactive(arr, cmap='gray',init_position=None):
    if init_position is None:
        init_position = [arr.shape[0] // 2, arr.shape[1] // 2, arr.shape[2] // 2]

    # Increase sliders width and description width (can be adjusted '800px' or changed to percentage '80%')
    slider_layout = widgets.Layout(width='80%')
    desc_style = {'description_width': '90px'}

    z_slider = widgets.IntSlider(value=init_position[0], min=0, max=arr.shape[0] - 1, step=1,
                                 description='Z (Axial)', layout=slider_layout, style=desc_style)
    y_slider = widgets.IntSlider(value=init_position[1], min=0, max=arr.shape[1] - 1, step=1,
                                 description='Y (Coronal)', layout=slider_layout, style=desc_style)
    x_slider = widgets.IntSlider(value=init_position[2], min=0, max=arr.shape[2] - 1, step=1,
                                 description='X (Sagittal)', layout=slider_layout, style=desc_style)

    ui = widgets.VBox([z_slider, y_slider, x_slider])
    out = widgets.interactive_output(
        lambda z, y, x: visualize_slices_at(arr, [z, y, x], cmap=cmap, line_width=1.5, highlight_center=True),
        {"z": z_slider, "y": y_slider, "x": x_slider}
    )

    display(ui, out)


# New functions: crop + zero padding + optional maximum inscribed sphere
def crop_centered_cube(arr, center, size, inscribed_sphere=False):
    # crop a cubic patch centered at `center` with side length `size` (int).
    # - arr: numpy array with shape (Z,Y,X)
    # - center: (z,y,x) in array indices
    # - size: int (same for all dims)
    # - inscribed_sphere: if True, zero out voxels outside the largest inscribed sphere
    # Returns cropped array of shape (size, size, size)
    assert size >= 1 and isinstance(size, int), 'size must be positive int'
    cz, cy, cx = center
    half = size // 2

    # define output array
    out = np.zeros((size, size, size), dtype=arr.dtype)

    # compute source and destination slices
    src_z0 = cz - half
    src_y0 = cy - half
    src_x0 = cx - half
    src_z1 = src_z0 + size
    src_y1 = src_y0 + size
    src_x1 = src_x0 + size

    # clipping to source image bounds
    iz0 = max(0, src_z0)
    iy0 = max(0, src_y0)
    ix0 = max(0, src_x0)
    iz1 = min(arr.shape[0], src_z1)
    iy1 = min(arr.shape[1], src_y1)
    ix1 = min(arr.shape[2], src_x1)

    # corresponding destination indices
    dz0 = iz0 - src_z0
    dy0 = iy0 - src_y0
    dx0 = ix0 - src_x0
    dz1 = dz0 + (iz1 - iz0)
    dy1 = dy0 + (iy1 - iy0)
    dx1 = dx0 + (ix1 - ix0)

    # copy overlap region
    out[dz0:dz1, dy0:dy1, dx0:dx1] = arr[iz0:iz1, iy0:iy1, ix0:ix1]

    if inscribed_sphere:
        # create spherical mask centered in output array
        grid_z, grid_y, grid_x = np.ogrid[0:size, 0:size, 0:size]
        center_coord = ((size - 1) / 2.0, (size - 1) / 2.0, (size - 1) / 2.0)
        dist2 = (grid_z - center_coord[0])**2 + (grid_y - center_coord[1])**2 + (grid_x - center_coord[2])**2
        radius = size / 2.0
        mask = dist2 <= (radius**2)
        out = out * mask.astype(out.dtype)

    return out


# save numpy three-dimensional array as nii.gz
def save_array_as_nifti(arr, reference=None, out_path='output.nii.gz'):
    # - arr: numpy array (Z,Y,X)
    # - reference: optional SimpleITK.Image or dict with spacing/origin/direction to copy metadata
    # - out_path: output filename (e.g., 'crop.nii.gz')
    
    # convert to sitk image (GetImageFromArray assumes arr is (z,y,x))
    sitk_img = sitk.GetImageFromArray(arr)

    if reference is not None:
        # accept SimpleITK.Image or dict-like
        if isinstance(reference, sitk.Image):
            sitk_img.SetSpacing(reference.GetSpacing())
            sitk_img.SetOrigin(reference.GetOrigin())
            try:
                sitk_img.SetDirection(reference.GetDirection())
            except Exception:
                pass
        elif isinstance(reference, dict):
            if 'spacing' in reference:
                sitk_img.SetSpacing(reference['spacing'])
            if 'origin' in reference:
                sitk_img.SetOrigin(reference['origin'])

    sitk.WriteImage(sitk_img, out_path)
    return out_path


- 了解医学影像空间位置

In [ ]:
# Get image infos
med_img, med_info = get_image_info(med_img_path)
print("Original Image Info:", med_info)

- 各向同性处理

In [ ]:
min_spacing = min(med_info["spacing"])
print(f"Minimum spacing: {min_spacing}")
med_arr = sitk.GetArrayFromImage(med_img)
med_arr_isotropic = set_isotropy(med_arr, med_info["spacing"], new_spacing=min_spacing)
print("Isotropic shape:", med_arr_isotropic.shape)

- 可视化中心切片

In [ ]:
# visualize center slices in three directions of the isotropic image
visualize_slices(med_arr_isotropic, cmap='jet', has_colorbar=False)

- 交互式可视化指定中心position的切片

In [ ]:
visualize_slices_at_interactive(med_arr_isotropic, cmap='jet', init_position=[35, 110, 55])

- 确定中心切片显示

In [ ]:
visualize_slices_at(med_arr_isotropic, position=[35, 110, 55], cmap='jet', add_line=False, highlight_center=False, has_labels=False)

- 裁剪图像

In [ ]:
cropped_data = crop_centered_cube(med_arr_isotropic, center=[35, 110, 55], size=80, inscribed_sphere=True)
print("Cropped data shape:", cropped_data.shape)
# visualize the cropped patch
visualize_slices(cropped_data, cmap='jet', has_colorbar=False)

- 保存图像

In [ ]:
# save the cropped patch as nii.gz
save_path = save_array_as_nifti(cropped_data, reference=med_img, out_path=cropped_save_path)
print(f"Cropped patch saved to: {save_path}")